# Laboratorul 2: predicții și dashboard

Continuare pentru setul curățat din Laboratorul 1. Codespaces pregătește și transferă un subset de aproximativ 100 MiB; Databricks Free Edition rulează Delta Lake, modelul MLlib și demonstrația de recuperare. Nu reexecutați curățarea sursei și nu utilizați directoarele experimentale `data/delta` sau `data/processed_delta` ca destinație pentru rezultate noi.

**Flux:** rulați celulele cu `LAB2_PHASE='codespaces'` în Codespaces; importați acest notebook în workspace-ul Databricks, selectați `LAB2_PHASE='databricks'` și rulați celulele cloud. Filesystem-ul celor două medii este separat. Kafka și MongoDB nu sunt folosite în acest laborator.

## 1. Configurare și verificarea mediului

În Codespaces, configurați `DATABRICKS_HOST`, `DATABRICKS_API` și opțional `DATABRICKS_VOLUME_PATH` prin Codespaces Secrets sau în fișierul local `.env` ignorat de Git. `DATABRICKS_API` este un token bearer. În Databricks, configurați widgetul `LAB2_PHASE` la `databricks`. Celulele nu afișează secretele și nu instalează pachete în timpul execuției.

In [2]:
import hashlib
import json
import os
import platform
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None:
    current_directory = Path.cwd().resolve()
    project_root = next(
        (candidate for candidate in (current_directory, *current_directory.parents)
         if (candidate / 'Laboratorul 1.ipynb').is_file()),
        current_directory,
    )
    load_dotenv(project_root / '.env', override=False)
else:
    project_root = Path.cwd().resolve()

try:
    LAB2_PHASE = dbutils.widgets.get('LAB2_PHASE')
except NameError:
    LAB2_PHASE = os.environ.get('LAB2_PHASE', 'codespaces').strip().lower()
except Exception:
    dbutils.widgets.dropdown('LAB2_PHASE', 'databricks', ['databricks', 'codespaces'])
    LAB2_PHASE = dbutils.widgets.get('LAB2_PHASE')

if LAB2_PHASE not in {'codespaces', 'databricks'}:
    raise ValueError('LAB2_PHASE trebuie să fie codespaces sau databricks.')

print(f'Python: {platform.python_version()}')
print(f'Faza notebookului: {LAB2_PHASE}')
print(f'Platformă: {platform.platform()}')
if LAB2_PHASE == 'codespaces':
    print('Proiect: configurație locală încărcată; tokenul nu este afișat.')


Python: 3.12.11
Faza notebookului: codespaces
Platformă: Linux-6.8.0-1064-azure-x86_64-with-glibc2.36
Proiect: configurație locală încărcată; tokenul nu este afișat.


In [4]:
from importlib.metadata import PackageNotFoundError, version

if LAB2_PHASE == 'codespaces':
    for package_name in ('pyspark', 'delta-spark', 'requests', 'python-dotenv'):
        try:
            package_version = version(package_name)
        except PackageNotFoundError as exc:
            raise RuntimeError(
                f'Lipsește {package_name}; reconstruiți Codespace-ul după instalarea requirements-codespaces.txt.'
            ) from exc
        print(f'{package_name}: {package_version}')
else:
    print(f'Databricks Runtime Spark: {spark.version}')


pyspark: 4.2.0
delta-spark: 4.4.0
requests: 2.32.5
python-dotenv: 1.1.1


## 2. Pregătirea subsetului și transferul din Codespaces

Rulați această secțiune în Codespaces după ce Lab 1 a produs `data/processed_delta`. Se selectează un eșantion reproductibil după hash-ul ID-ului și se calibrează dimensiunea Parquet în jurul a 100 MiB. Fișierele și manifestul rămân sub `data/lab2/`, exclus din Git.

Înainte de upload, creați o singură dată Unity Catalog Volume în Databricks cu numele care corespunde lui `DATABRICKS_VOLUME_PATH`. Pentru valorile implicite, rulați în SQL Editor:

```sql
CREATE SCHEMA IF NOT EXISTS workspace.default;
CREATE VOLUME IF NOT EXISTS workspace.default.lab2_files;
```

Alternativ, creați volumul în Catalog Explorer. Dacă Free Edition folosește alt catalog/schema sau tokenul nu permite Files API, modificați widgeturile/secretul și folosiți uploadul manual în UI.

In [5]:
if LAB2_PHASE == 'codespaces':
    import requests
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession, functions as F

    source_path = project_root / 'data' / 'processed_delta'
    if not source_path.is_dir():
        raise FileNotFoundError(
            f'Nu există {source_path}. Rulați Laboratorul 1 până la salvarea data/processed_delta.'
        )

    spark_builder = (
        SparkSession.builder.appName('Laboratorul_2_Handoff')
        .master('local[*]')
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    )
    spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()
    source_df = spark.read.format('delta').load(str(source_path))
    if 'ID' not in source_df.columns:
        raise ValueError('Setul procesat nu conține coloana ID necesară selecției stabile.')

    target_bytes = 100 * 1024 * 1024
    hash_modulus = 10_000
    hash_cutoff = 1_000
    handoff_root = project_root / 'data' / 'lab2' / 'handoff'
    parquet_dir = handoff_root / 'parquet'
    parquet_dir.parent.mkdir(parents=True, exist_ok=True)

    def directory_bytes(directory):
        return sum(path.stat().st_size for path in directory.rglob('*') if path.is_file())

    for attempt in range(1, 6):
        sample_df = source_df.filter(
            source_df['ID'].isNotNull()
            & (F.pmod(F.xxhash64('ID'), F.lit(hash_modulus)) < hash_cutoff)
        )
        sample_df.write.mode('overwrite').parquet(str(parquet_dir))
        parquet_bytes = directory_bytes(parquet_dir)
        if parquet_bytes == 0:
            raise RuntimeError('Eșantionul Parquet este gol; verificați coloana ID.')
        if 0.9 * target_bytes <= parquet_bytes <= 1.1 * target_bytes:
            break
        hash_cutoff = max(1, min(hash_modulus - 1, round(hash_cutoff * target_bytes / parquet_bytes)))
    else:
        print('Atenție: dimensiunea Parquet nu a intrat în toleranța de ±10% după 5 calibrări.')

    sample_count = sample_df.count()
    parquet_files = sorted(path for path in parquet_dir.glob('*.parquet') if path.is_file())
    if not parquet_files:
        raise RuntimeError('Nu au fost create fișiere Parquet pentru transfer.')

    manifest = {
        'source': 'data/processed_delta',
        'selection': f'pmod(xxhash64(ID), {hash_modulus}) < {hash_cutoff}; ID IS NOT NULL',
        'target_bytes': target_bytes,
        'parquet_bytes': parquet_bytes,
        'row_count': sample_count,
        'schema': sample_df.schema.jsonValue(),
        'files': [],
    }
    for parquet_file in parquet_files:
        digest = hashlib.sha256()
        with parquet_file.open('rb') as file_handle:
            for chunk in iter(lambda: file_handle.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        manifest['files'].append({
            'name': parquet_file.name,
            'size_bytes': parquet_file.stat().st_size,
            'sha256': digest.hexdigest(),
        })

    manifest_path = handoff_root / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Rânduri selectate: {sample_count:,}')
    print(f'Dimensiune Parquet: {parquet_bytes / (1024 ** 2):.2f} MiB')
    print(f'Fișiere Parquet: {len(parquet_files)}; schema capturată în manifest.')

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7134069b-4782-4a40-91b0-1c88725c0b3a;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.0 in

Rânduri selectate: 1,379,481
Dimensiune Parquet: 97.20 MiB
Fișiere Parquet: 6; schema capturată în manifest.


In [6]:
if LAB2_PHASE == 'codespaces':
    manifest['schema'] = [
        {'name': field.name, 'type': field.dataType.simpleString()}
        for field in sample_df.schema.fields
    ]
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')


In [7]:
if LAB2_PHASE == 'codespaces':
    databricks_host = os.environ.get('DATABRICKS_HOST', '').strip().rstrip('/')
    databricks_token = os.environ.get('DATABRICKS_API', '').strip()
    volume_path = os.environ.get(
        'DATABRICKS_VOLUME_PATH',
        '/Volumes/workspace/default/lab2_files',
    ).strip().rstrip('/')
    if not databricks_host or not databricks_token:
        raise RuntimeError('Configurați DATABRICKS_HOST și DATABRICKS_API în Codespaces Secrets sau .env.')
    if not databricks_host.startswith('https://'):
        raise ValueError('DATABRICKS_HOST trebuie să fie URL HTTPS al workspace-ului.')
    if not volume_path.startswith('/Volumes/'):
        raise ValueError('DATABRICKS_VOLUME_PATH trebuie să înceapă cu /Volumes/.')

    from urllib.parse import quote

    api_headers = {'Authorization': f'Bearer {databricks_token}'}

    def files_api_url(endpoint, remote_path):
        return f'{databricks_host}/api/2.0/fs/{endpoint}{quote(remote_path, safe="/")}'

    remote_input_dir = f'{volume_path}/lab2-input'
    directory_response = requests.put(
        files_api_url('directories', remote_input_dir),
        headers=api_headers,
        timeout=(15, 60),
    )
    if directory_response.status_code not in (200, 201, 204):
        raise RuntimeError(
            f'Crearea directorului Volume a eșuat (HTTP {directory_response.status_code}). '
            'Verificați existența UC Volume și permisiunile tokenului.'
        )

    upload_files = [*parquet_files, manifest_path]
    for local_file in upload_files:
        remote_file = f'{remote_input_dir}/{local_file.name}'
        file_size = local_file.stat().st_size
        if file_size > 5 * 1024 ** 3:
            raise ValueError(f'{local_file.name} depășește limita Files API de 5 GiB.')
        upload_headers = {
            **api_headers,
            'Content-Type': 'application/octet-stream',
            'Content-Length': str(file_size),
        }
        with local_file.open('rb') as file_handle:
            response = requests.put(
                files_api_url('files', remote_file),
                headers=upload_headers,
                data=file_handle,
                timeout=(20, 900),
            )
        if response.status_code not in (200, 201, 204):
            raise RuntimeError(
                f'Uploadul {local_file.name} a eșuat (HTTP {response.status_code}); '
                'verificați tokenul, Files API și permisiunile asupra Volume-ului.'
            )
        metadata = requests.head(
            files_api_url('files', remote_file),
            headers=api_headers,
            timeout=(15, 60),
        )
        remote_size = int(metadata.headers.get('Content-Length', '-1'))
        if metadata.status_code != 200 or remote_size != file_size:
            raise RuntimeError(
                f'Verificarea dimensiunii pentru {local_file.name} a eșuat; '
                'reîncercați uploadul înainte de a porni notebookul Databricks.'
            )
        print(f'Upload verificat: {local_file.name} ({file_size:,} bytes)')
    print(f'Path Databricks: {remote_input_dir}')


Upload verificat: part-00000-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (18,141,838 bytes)
Upload verificat: part-00001-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (18,153,580 bytes)
Upload verificat: part-00002-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (18,163,447 bytes)
Upload verificat: part-00003-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (18,193,209 bytes)
Upload verificat: part-00004-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (18,132,432 bytes)
Upload verificat: part-00005-c234771e-ad34-4895-a5ef-f74aac1fbb64-c000.snappy.parquet (10,349,460 bytes)
Upload verificat: manifest.json (3,178 bytes)
Path Databricks: /Volumes/workspace/default/lab2_files/lab2-input


## 3. Delta Lake și import în Databricks Free

Importați notebookul în workspace-ul Databricks după ce Codespaces a încărcat datele. În Databricks, widgeturile aleg catalogul, schema și volumul; valorile implicite sunt `workspace.default.lab2_files`. Dacă acest catalog/schema nu sunt disponibile, selectați valorile permise în workspace. Celula de import verifică manifestul, apoi scrie și recitește o tabelă Delta dedicată.

In [13]:
print(LAB2_PHASE)
if LAB2_PHASE == 'databricks':
    import re

    from delta.tables import DeltaTable
    from pyspark.ml import Pipeline
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
    from pyspark.sql import functions as F
    from pyspark.ml.functions import vector_to_array

    widget_defaults = {
        'LAB2_CATALOG': 'workspace',
        'LAB2_SCHEMA': 'default',
        'LAB2_VOLUME': 'lab2_files',
    }
    for widget_name, widget_default in widget_defaults.items():
        try:
            dbutils.widgets.get(widget_name)
        except Exception:
            dbutils.widgets.text(widget_name, widget_default)

    catalog_name = dbutils.widgets.get('LAB2_CATALOG').strip()
    schema_name = dbutils.widgets.get('LAB2_SCHEMA').strip()
    volume_name = dbutils.widgets.get('LAB2_VOLUME').strip()
    for identifier in (catalog_name, schema_name, volume_name):
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', identifier):
            raise ValueError('Catalogul, schema și volumul trebuie să fie identificatori SQL simpli.')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog_name}`.`{schema_name}`.`{volume_name}`')
    volume_root = f'/Volumes/{catalog_name}/{schema_name}/{volume_name}'
    input_directory = f'{volume_root}/lab2-input'
    manifest_file = f'{input_directory}/manifest.json'

    with open(manifest_file, 'r', encoding='utf-8') as manifest_handle:
        transfer_manifest = json.load(manifest_handle)
    imported_df = spark.read.parquet(input_directory)
    imported_count = imported_df.count()
    imported_schema = [
        {'name': field.name, 'type': field.dataType.simpleString()}
        for field in imported_df.schema.fields
    ]
    if imported_count != transfer_manifest['row_count']:
        raise AssertionError('Numărul de rânduri din Databricks nu corespunde manifestului.')
    if imported_schema != transfer_manifest['schema']:
        raise AssertionError('Schema Parquet din Databricks nu corespunde manifestului.')

    INPUT_TABLE = f'{catalog_name}.{schema_name}.lab2_input'
    (imported_df.write.format('delta')
     .option('delta.columnMapping.mode', 'name')
     .option('delta.minReaderVersion', '2')
     .option('delta.minWriterVersion', '5')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(INPUT_TABLE))
    source_df = spark.table(INPUT_TABLE)
    assert source_df.count() == imported_count
    assert source_df.schema == imported_df.schema
    source_df.printSchema()
    source_df.show(5, truncate=False)
    print(f'Tabelă Delta verificată: {INPUT_TABLE}; {imported_count:,} rânduri.')

codespaces


## 4. Clasificare MLlib: probabilitatea arestării

Ținta este `Arrest`. Modelul folosește doar atribute cunoscute la momentul predicției: tipul incidentului, districtul, câteva caracteristici temporale din `Date` și indicatorul `Domestic`. `ID` este păstrat numai ca cheie, iar `Arrest` nu intră în features. Split-ul este reproductibil; raportăm două metrici pe test și un singur grafic ROC. Relația statistică nu implică o recomandare operațională sau cauzalitate.

In [12]:
print(LAB2_PHASE)
if LAB2_PHASE == 'databricks':
    model_input = (
        source_df
        .filter(F.col('Arrest').isNotNull() & F.col('Date').isNotNull())
        .withColumn('label', F.col('Arrest').cast('double'))
        .withColumn('event_year', F.year('Date').cast('double'))
        .withColumn('event_month', F.month('Date').cast('double'))
        .withColumn('event_hour', F.hour('Date').cast('double'))
        .withColumn('event_weekday', F.dayofweek('Date').cast('double'))
        .withColumn('domestic_flag', F.coalesce(F.col('Domestic').cast('double'), F.lit(0.0)))
        .withColumn('primary_type_feature', F.coalesce(F.col('Primary Type'), F.lit('Unknown')))
        .withColumn('district_feature', F.coalesce(F.col('District').cast('string'), F.lit('Unknown')))
        .select(
            'ID', 'Date', 'Year', 'Primary Type', 'District', 'Arrest', 'label',
            'event_year', 'event_month', 'event_hour', 'event_weekday', 'domestic_flag',
            'primary_type_feature', 'district_feature',
        )
    )
    class_distribution = model_input.groupBy('label').count().orderBy('label')
    class_distribution.show()
    if class_distribution.count() != 2:
        raise ValueError('Clasificarea necesită ambele clase Arrest=False și Arrest=True.')

    train_df, test_df = model_input.randomSplit([0.8, 0.2], seed=42)
    if train_df.select('label').distinct().count() != 2 or test_df.select('label').distinct().count() != 2:
        raise ValueError('Split-ul nu conține ambele clase; verificați balanța sau ajustați seed-ul.')

    category_columns = ['primary_type_feature', 'district_feature']
    indexed_columns = ['primary_type_index', 'district_index']
    encoded_columns = ['primary_type_ohe', 'district_ohe']
    indexer = StringIndexer(
        inputCols=category_columns,
        outputCols=indexed_columns,
        handleInvalid='keep',
    )
    encoder = OneHotEncoder(
        inputCols=indexed_columns,
        outputCols=encoded_columns,
        handleInvalid='keep',
    )
    numeric_columns = ['event_year', 'event_month', 'event_hour', 'event_weekday', 'domestic_flag']
    assembler = VectorAssembler(
        inputCols=numeric_columns + encoded_columns,
        outputCol='features',
        handleInvalid='error',
    )
    classifier = LogisticRegression(
        featuresCol='features',
        labelCol='label',
        maxIter=30,
        regParam=0.01,
    )
    arrest_pipeline = Pipeline(stages=[indexer, encoder, assembler, classifier])
    arrest_model = arrest_pipeline.fit(train_df)
    test_predictions = arrest_model.transform(test_df).cache()
    test_count = test_predictions.count()

    auc_evaluator = BinaryClassificationEvaluator(
        labelCol='label',
        rawPredictionCol='rawPrediction',
        metricName='areaUnderROC',
    )
    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='f1',
    )
    test_auc = auc_evaluator.evaluate(test_predictions)
    test_f1 = f1_evaluator.evaluate(test_predictions)
    print(f'Test records: {test_count:,}')
    print(f'Area under ROC: {test_auc:.4f}')
    print(f'F1: {test_f1:.4f}')

    model_path = f'{volume_root}/models/arrest_logistic_regression'
    arrest_model.write().overwrite().save(model_path)
    print(f'Model salvat în Volume: {model_path}')

codespaces


In [10]:
if LAB2_PHASE == 'databricks':
    import matplotlib.pyplot as plt
    import numpy as np

    roc_sample = (
        test_predictions
        .select('ID', 'label', vector_to_array('probability')[1].alias('score'))
        .orderBy(F.xxhash64('ID'))
        .limit(50_000)
        .toPandas()
    )
    roc_labels = roc_sample['label'].to_numpy(dtype=int)
    roc_scores = roc_sample['score'].to_numpy(dtype=float)
    if len(np.unique(roc_labels)) != 2:
        raise ValueError('Eșantionul ROC trebuie să conțină ambele clase.')

    order = np.argsort(-roc_scores, kind='mergesort')
    sorted_scores = roc_scores[order]
    sorted_labels = roc_labels[order]
    threshold_indices = np.r_[np.where(np.diff(sorted_scores))[0], len(sorted_scores) - 1]
    true_positive_rate = np.r_[0.0, np.cumsum(sorted_labels)[threshold_indices] / sorted_labels.sum()]
    negative_count = len(sorted_labels) - sorted_labels.sum()
    false_positive_rate = np.r_[
        0.0,
        (np.arange(1, len(sorted_labels) + 1) - np.cumsum(sorted_labels))[threshold_indices]
        / negative_count,
    ]

    plt.figure(figsize=(7, 5))
    plt.plot(false_positive_rate, true_positive_rate, color='#247ba0', label='Model MLlib')
    plt.plot([0, 1], [0, 1], linestyle='--', color='#777777', label='Referință aleatoare')
    plt.title('Curba ROC pentru clasificarea Arrest')
    plt.xlabel('Rata fals pozitive')
    plt.ylabel('Rata adevărate pozitive')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    print(
        f'Interpretare: AUC={test_auc:.3f} măsoară separarea claselor; '
        'graficul nu stabilește cauzalitate și nu trebuie folosit singur pentru decizii individuale.'
    )


## 5. Predicții, întrerupere controlată și reluare

Predicțiile sunt scrise în Delta în loturi determinate prin hash-ul lui `incident_id`. După primul `MERGE` injectăm deliberat o întrerupere la nivelul aplicației; reluarea procesează din nou ambele loturi cu upsert idempotent. Aceasta demonstrează reluarea logicii și evitarea dublurilor, nu simularea unei căderi fizice de cluster sau garanții complete de disaster recovery.

In [11]:
if LAB2_PHASE == 'databricks':
    from datetime import datetime, timezone
    from pyspark.sql import types as T

    run_id = __import__('uuid').uuid4().hex
    run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    all_predictions = (
        arrest_model.transform(model_input)
        .select(
            F.col('ID').cast('long').alias('incident_id'),
            F.year('Date').cast('int').alias('event_year'),
            F.col('Primary Type').alias('primary_type'),
            F.col('District').cast('string').alias('district'),
            F.col('label').cast('int').alias('actual_arrest'),
            F.col('prediction').cast('int').alias('predicted_arrest'),
            vector_to_array('probability')[1].cast('double').alias('predicted_arrest_probability'),
        )
        .cache()
    )
    expected_prediction_count = all_predictions.count()
    PREDICTIONS_TABLE = f'{catalog_name}.{schema_name}.lab2_arrest_predictions'
    BACKUP_TABLE = f'{catalog_name}.{schema_name}.lab2_arrest_predictions_backup'
    AUDIT_TABLE = f'{catalog_name}.{schema_name}.lab2_run_audit'

    # Aceste tabele dedicate sunt artefacte generate de laborator și sunt recreate la rularea demonstrației.
    spark.sql(f'DROP TABLE IF EXISTS {PREDICTIONS_TABLE}')
    all_predictions.limit(0).write.format('delta').mode('overwrite').saveAsTable(PREDICTIONS_TABLE)
    target_table = DeltaTable.forName(spark, PREDICTIONS_TABLE)

    def merge_prediction_batch(batch_df):
        (target_table.alias('target')
         .merge(batch_df.alias('source'), 'target.incident_id = source.incident_id')
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())

    batches = [
        all_predictions.filter(F.pmod(F.xxhash64('incident_id'), F.lit(2)) == batch_number)
        for batch_number in (0, 1)
    ]
    batch_counts = [batch.count() for batch in batches]
    if sum(batch_counts) != expected_prediction_count:
        raise AssertionError('Partiționarea deterministă a loturilor a pierdut rânduri.')

    merge_prediction_batch(batches[0])
    try:
        raise RuntimeError('Întrerupere controlată după commit-ul primului lot.')
    except RuntimeError as failure:
        if not str(failure).startswith('Întrerupere controlată'):
            raise
        print('Întreruperea controlată a fost injectată după primul MERGE; începe reluarea idempotentă.')

    for batch in batches:
        merge_prediction_batch(batch)

    prediction_table_df = spark.table(PREDICTIONS_TABLE)
    actual_prediction_count = prediction_table_df.count()
    duplicate_id_count = (
        prediction_table_df.groupBy('incident_id').count()
        .filter(F.col('count') > 1)
        .limit(1)
        .count()
    )
    missing_id_count = (
        all_predictions.select('incident_id').distinct()
        .join(prediction_table_df.select('incident_id').distinct(), 'incident_id', 'left_anti')
        .limit(1)
        .count()
    )
    extra_id_count = (
        prediction_table_df.select('incident_id').distinct()
        .join(all_predictions.select('incident_id').distinct(), 'incident_id', 'left_anti')
        .limit(1)
        .count()
    )
    assert actual_prediction_count == expected_prediction_count
    assert duplicate_id_count == missing_id_count == extra_id_count == 0
    print(
        f'Recovery verificat: {actual_prediction_count:,} predicții; '
        '0 ID duplicate, lipsă sau suplimentare.'
    )

    spark.sql(f'DROP TABLE IF EXISTS {BACKUP_TABLE}')
    (prediction_table_df.write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(BACKUP_TABLE))
    backup_df = spark.table(BACKUP_TABLE)
    assert backup_df.count() == expected_prediction_count
    assert backup_df.schema == prediction_table_df.schema
    print(f'Backup Delta verificat: {BACKUP_TABLE}; {backup_df.count():,} rânduri.')

    run_finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    audit_schema = T.StructType([
        T.StructField('run_id', T.StringType(), False),
        T.StructField('started_at_utc', T.TimestampType(), False),
        T.StructField('finished_at_utc', T.TimestampType(), False),
        T.StructField('expected_rows', T.LongType(), False),
        T.StructField('written_rows', T.LongType(), False),
        T.StructField('failure_injected', T.BooleanType(), False),
        T.StructField('status', T.StringType(), False),
    ])
    audit_row = [(
        run_id, run_started_at, run_finished_at,
        expected_prediction_count, actual_prediction_count, True, 'recovered',
    )]
    (spark.createDataFrame(audit_row, audit_schema)
     .write.format('delta').mode('append').saveAsTable(AUDIT_TABLE))
    spark.table(AUDIT_TABLE).orderBy(F.col('started_at_utc').desc()).show(5, truncate=False)
    all_predictions.unpersist()


## 6. Set de date pentru Power BI și raport în browser

Tabelul pregătit include câmpurile necesare pentru predicții, KPI și filtre. În Power BI Service creați mai întâi o conexiune la Databricks SQL din **My workspace**, selectați `lab2_powerbi_dashboard` și salvați raportul. Testați înainte dacă licența Free permite conexiunea, refresh-ul și descărcarea `.pbix`; încărcarea unui CSV local nu este fallback pentru livrabilul PBIX. Documentația Microsoft poate impune capacitate Premium pentru publicarea directă din Databricks către Power BI; fluxul de conectare manuală se verifică în UI-ul contului.

In [ ]:
if LAB2_PHASE == 'databricks':
    DASHBOARD_TABLE = f'{catalog_name}.{schema_name}.lab2_powerbi_dashboard'
    dashboard_df = (
        prediction_table_df
        .withColumn('prediction_run_id', F.lit(run_id))
        .withColumn('scored_at_utc', F.current_timestamp())
        .select(
            'incident_id', 'event_year', 'primary_type', 'district',
            'actual_arrest', 'predicted_arrest', 'predicted_arrest_probability',
            'prediction_run_id', 'scored_at_utc',
        )
    )
    (dashboard_df.write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(DASHBOARD_TABLE))
    dashboard_table_df = spark.table(DASHBOARD_TABLE)
    assert dashboard_table_df.count() == expected_prediction_count
    dashboard_table_df.printSchema()
    dashboard_table_df.show(5, truncate=False)
    print(f'Sursă pregătită pentru Power BI: {DASHBOARD_TABLE}')

### Structura propusă a raportului

Conectați Power BI Service la endpoint-ul Databricks SQL folosind **Server Hostname** și **HTTP Path** din setările SQL Warehouse; selectați `lab2_powerbi_dashboard` în modul Import dacă este disponibil. Folosiți `prediction_run_id` și `scored_at_utc` pentru a demonstra refresh-ul. Păstrați raportul în același workspace/semantic model și testați **File → Download this file** înainte de a considera `.pbix` valid.

Măsuri DAX de pornire:

```DAX
Incidente evaluate = COUNTROWS(lab2_powerbi_dashboard)
Arestări prezise = SUM(lab2_powerbi_dashboard[predicted_arrest])
Probabilitate medie prezisă = AVERAGE(lab2_powerbi_dashboard[predicted_arrest_probability])
```

- KPI: `Incidente evaluate` și `Arestări prezise` (sau probabilitatea medie).
- Grafic cu coloane: arestări prezise după `primary_type`. Grafic liniar: probabilitatea medie prezisă după `event_year`.
- Filtre interactive: `event_year` și `district`.
- Decizie posibilă 1: direcționați verificarea calității datelor către categorii/districte unde rata reală și probabilitatea estimată diferă mult; nu interpretați scorul drept evaluare individuală.
- Decizie posibilă 2: analizați schimbările anuale ale volumului prezis înainte de planificarea resurselor de analiză, confirmându-le cu datele reale.

Modelul reflectă datele istorice raportate, dezechilibrul claselor și diferențe de raportare între districte; scorul nu este cauzal și nu trebuie folosit pentru decizii punitive sau automatizate. Un refresh schimbă `prediction_run_id`/`scored_at_utc`; dacă sursa de date este aceeași, valorile de predicție pot rămâne identice.

## 7. Monitorizare, salvare și predare

Backup-ul este o copie Delta separată pentru recuperare; nu este sharding (împărțirea datelor între noduri) și nici replicare automată între servere. `lab2_run_audit` păstrează run ID, intervalul UTC, numărul așteptat/scris și statusul recuperării. Pentru dovada monitorizării, păstrați și capturi din istoricul run-ului și Spark UI/SQL query profile disponibile în Databricks Free.

La final:
1. Rulați notebookul în Databricks și verificați toate assert-urile, modelul salvat în Volume, tabelele Delta, auditul și tabelul `lab2_powerbi_dashboard`.
2. Exportați notebookul **cu rezultate** din Databricks în `.ipynb`, apoi generați HTML-ul pentru predare. Nu folosiți un export fără outputuri ca dovadă a rulării.
3. Salvați raportul `.pbix` din Power BI Service dacă opțiunea este disponibilă pentru această conexiune; altfel consemnați limitarea și folosiți Power BI Desktop gratuit pentru cerința PBIX din ghid.
4. Păstrați CSV/Parquet, manifestele și exporturile sub `data/`; nu comiteți date mari, `.env`, tokenuri sau outputuri care conțin secrete.

Nu se pornesc servicii Kafka/MongoDB și nu se repetă benchmarkurile Laboratorului 1.